In [1]:
import os
import uuid

import httpx
from langchain_openai import ChatOpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "")
CHAT_MODEL = os.getenv("CHAT_MODEL", "")

AGENT_URL = os.getenv("AGENT_URL", "http://localhost:8002")
FAISS_URL = os.getenv("FAISS_URL", "http://localhost:8000")

llm = ChatOpenAI(model=CHAT_MODEL, base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)

# RAG Pipeline Evaluation

Evaluates the vuln-chat RAG pipeline using LLM-as-judge metrics:
- **Faithfulness**: does the answer contradict the retrieved context?
- **Groundedness**: is the answer derived only from the retrieved context?
- **Answer Relevancy**: does the answer directly address the question?
- **Context Precision**: are the retrieved chunks relevant to the question?
- **Context Recall**: does the retrieved context contain everything needed?

Test cases are loaded from `testset.json` (generated by `generate_testset.ipynb`).

### Agent query helper

In [2]:
def query_agent(question: str, retriever_mode: str = "faiss") -> dict:
    """Call the agent's sync endpoint. Returns response + retrieved contexts."""
    resp = httpx.post(
        f"{AGENT_URL}/chat/sync",
        json={"message": question, "session_id": str(uuid.uuid4()), "retriever_mode": retriever_mode},
        timeout=120.0,
    )
    resp.raise_for_status()
    data = resp.json()
    return {"response": data.get("response", ""), "retrieved_contexts": data.get("retrieved_docs", [])}

### Load test cases

In [3]:
import json

with open("testset.json") as f:
    testset = json.load(f)

all_cases = []
for source in ("manual", "synthetic"):
    for test_case in testset[source]:
        test_case["source"] = source
        all_cases.append(test_case)

print(f"Loaded {len(all_cases)} total test cases")

Loaded 14 total test cases


## Run FAISS pipeline on all test cases

In [4]:
from evaluators import EvalSample, evaluate

eval_samples = []

for i, test_case in enumerate(all_cases):
    print(f"  [{i+1}/{len(all_cases)}] {test_case['source']}: {test_case['user_input'][:60]}...")
    result = query_agent(test_case["user_input"])
    eval_samples.append(EvalSample(
        user_input=test_case["user_input"],
        reference=test_case["reference"],
        response=result["response"],
        retrieved_contexts=result["retrieved_contexts"],
        source=test_case["source"],
    ))

  [1/14] manual: What is the CVSS score and attack vector for Log4Shell?...
  [2/14] manual: What remediation steps are recommended for Spring4Shell?...
  [3/14] manual: What is the Heartbleed vulnerability and what does it expose...
  [4/14] manual: How does the runc container escape vulnerability work?...
  [5/14] manual: Which CVEs are related to OpenSSL X.509 certificate handling...
  [6/14] synthetic: What kind of impact does Heartbleed have?...
  [7/14] synthetic: What kind of impact can this Log4j issue have in containeriz...
  [8/14] synthetic: How urgent is CVE-2021-40539 to prioritize?...
  [9/14] synthetic: How severe is CVE-2023-23397 and can it be exploited over th...
  [10/14] synthetic: How severe is CVE-2022-1388 and can it be exploited remotely...
  [11/14] synthetic: Which incoming client-side or server-side issue should I esc...
  [12/14] synthetic: Which OpenSSL issue is the bigger patching priority for inte...
  [13/14] synthetic: Which of these enterprise software

## Evaluate FAISS

In [5]:
results = evaluate(llm=llm, samples=eval_samples)
results

Evaluating sample 1/14
Evaluating sample 2/14
Evaluating sample 3/14
Evaluating sample 4/14
Evaluating sample 5/14
Evaluating sample 6/14
Evaluating sample 7/14
Evaluating sample 8/14
Evaluating sample 9/14
Evaluating sample 10/14
Evaluating sample 11/14
Evaluating sample 12/14
Evaluating sample 13/14
Evaluating sample 14/14


[SampleResult(sample=EvalSample(user_input='What is the CVSS score and attack vector for Log4Shell?', response='Log4Shell refers to **CVE-2021-44228**.\n\n- **CVSS score:** **10.0** (Critical)\n- **Attack vector:** **NETWORK**\n\nThe CVE data describes it as a network-exploitable issue in Apache Log4j2 that can lead to remote code execution.', retrieved_contexts=['[score: 0.5599]\nCVE ID: CVE-2021-44228\nPublished: 2021-12-10T10:15:09.143\nStatus: Analyzed\nSeverity: CRITICAL (CVSS 10.0)\nAttack Vector: NETWORK\nCWE: CWE-20, CWE-400, CWE-502, CWE-917\nDescription: Apache Log4j2 2.0-beta9 through 2.15.0 (excluding security releases 2.12.2, 2.12.3, and 2.3.1) JNDI features used in configuration, log messages, and parameters do not protect against attacker controlled LDAP and other JNDI related endpoints. An attacker who can control log messages or log message parameters can execute arbitrary code loaded from LDAP servers when message lookup substitution is enabled. From log4j 2.15.0, thi

## Aggregate FAISS scores

In [6]:
metric_names = ["faithfulness", "groundedness", "answer_relevancy", "context_precision", "context_recall"]

scores = {metric_name: [] for metric_name in metric_names}
scores_by_source = {}

for sample_result in results:
    source = sample_result.sample.source
    if source not in scores_by_source:
        scores_by_source[source] = {metric_name: [] for metric_name in metric_names}
    for metric_name in metric_names:
        metric_result = getattr(sample_result, metric_name)
        if metric_result:
            scores[metric_name].append(metric_result.score)
            scores_by_source[source][metric_name].append(metric_result.score)

print("=== Overall ===")
for metric_name in metric_names:
    print(f"  {metric_name}: {sum(scores[metric_name]) / len(scores[metric_name]):.3f}")

print()
for source, source_scores in scores_by_source.items():
    print(f"=== {source} ===")
    for metric_name in metric_names:
        vals = source_scores[metric_name]
        print(f"  {metric_name}: {sum(vals) / len(vals):.3f}")

=== Overall ===
  faithfulness: 0.993
  groundedness: 0.656
  answer_relevancy: 0.882
  context_precision: 0.564
  context_recall: 0.679

=== manual ===
  faithfulness: 1.000
  groundedness: 0.810
  answer_relevancy: 0.984
  context_precision: 0.720
  context_recall: 0.900
=== synthetic ===
  faithfulness: 0.989
  groundedness: 0.571
  answer_relevancy: 0.826
  context_precision: 0.478
  context_recall: 0.556


## Export FAISS results

In [7]:
from datetime import datetime, timezone

def build_output(results, scores, scores_by_source, metric_names):
    return {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "overall": {metric_name: sum(scores[metric_name]) / len(scores[metric_name]) for metric_name in metric_names},
        "by_source": {
            source: {metric_name: sum(source_scores[metric_name]) / len(source_scores[metric_name]) for metric_name in metric_names}
            for source, source_scores in scores_by_source.items()
        },
        "samples": [
            {
                "user_input": result.sample.user_input,
                "reference": result.sample.reference,
                "response": result.sample.response,
                "retrieved_contexts": result.sample.retrieved_contexts,
                "source": result.sample.source,
                "scores": {
                    metric_name: {"score": getattr(result, metric_name).score, "reasoning": getattr(result, metric_name).reasoning}
                    for metric_name in metric_names
                    if getattr(result, metric_name)
                },
            }
            for result in results
        ],
    }

faiss_output = build_output(results, scores, scores_by_source, metric_names)
with open("eval_results_faiss.json", "w") as f:
    json.dump(faiss_output, f, indent=2)

print(f"Wrote {len(faiss_output['samples'])} sample results to eval_results_faiss.json")

Wrote 14 sample results to eval_results_faiss.json


## Run LightRAG pipeline on all test cases

In [8]:
lightrag_samples = []

for i, test_case in enumerate(all_cases):
    print(f"  [{i+1}/{len(all_cases)}] {test_case['source']}: {test_case['user_input'][:60]}...")
    result = query_agent(test_case["user_input"], retriever_mode="lightrag")
    lightrag_samples.append(EvalSample(
        user_input=test_case["user_input"],
        reference=test_case["reference"],
        response=result["response"],
        retrieved_contexts=result["retrieved_contexts"],
        source=test_case["source"],
    ))

  [1/14] manual: What is the CVSS score and attack vector for Log4Shell?...
  [2/14] manual: What remediation steps are recommended for Spring4Shell?...
  [3/14] manual: What is the Heartbleed vulnerability and what does it expose...
  [4/14] manual: How does the runc container escape vulnerability work?...
  [5/14] manual: Which CVEs are related to OpenSSL X.509 certificate handling...
  [6/14] synthetic: What kind of impact does Heartbleed have?...
  [7/14] synthetic: What kind of impact can this Log4j issue have in containeriz...
  [8/14] synthetic: How urgent is CVE-2021-40539 to prioritize?...
  [9/14] synthetic: How severe is CVE-2023-23397 and can it be exploited over th...
  [10/14] synthetic: How severe is CVE-2022-1388 and can it be exploited remotely...
  [11/14] synthetic: Which incoming client-side or server-side issue should I esc...
  [12/14] synthetic: Which OpenSSL issue is the bigger patching priority for inte...
  [13/14] synthetic: Which of these enterprise software

## Evaluate LightRAG

In [9]:
lightrag_results = evaluate(llm=llm, samples=lightrag_samples)
lightrag_results

Evaluating sample 1/14
Evaluating sample 2/14
Evaluating sample 3/14
Evaluating sample 4/14
Evaluating sample 5/14
Evaluating sample 6/14
Evaluating sample 7/14
Evaluating sample 8/14
Evaluating sample 9/14
Evaluating sample 10/14
Evaluating sample 11/14
Evaluating sample 12/14
Evaluating sample 13/14
Evaluating sample 14/14


[SampleResult(sample=EvalSample(user_input='What is the CVSS score and attack vector for Log4Shell?', response='Log4Shell, CVE-2021-44228, has a CVSS score of 10.0 and an attack vector of Network (AV:N).', retrieved_contexts=['## CVE-2021-44228 (Log4Shell)\n\n**CVE-2021-44228**, also known as **Log4Shell**, has a **CVSS score of 10.0** and an **attack vector of NETWORK**.\n\nThe vulnerability affects **Apache Log4j2** and is described as a critical issue in which **JNDI features** used in configuration, log messages, and parameters do not protect against attacker-controlled **LDAP** and other JNDI-related endpoints. According to the provided context, an attacker who can control log messages or log message parameters can achieve **arbitrary code execution** when message lookup substitution is enabled.\n\nThe context also states that this issue is **specific to log4j-core** and **does not affect** **log4net**, **log4cxx**, or other Apache Logging Services projects.\n\n### References\n\n-

## Aggregate LightRAG scores

In [10]:
lr_scores = {metric_name: [] for metric_name in metric_names}
lr_scores_by_source = {}

for sample_result in lightrag_results:
    source = sample_result.sample.source
    if source not in lr_scores_by_source:
        lr_scores_by_source[source] = {metric_name: [] for metric_name in metric_names}
    for metric_name in metric_names:
        metric_result = getattr(sample_result, metric_name)
        if metric_result:
            lr_scores[metric_name].append(metric_result.score)
            lr_scores_by_source[source][metric_name].append(metric_result.score)

print("=== Overall ===")
for metric_name in metric_names:
    print(f"  {metric_name}: {sum(lr_scores[metric_name]) / len(lr_scores[metric_name]):.3f}")

print()
for source, source_scores in lr_scores_by_source.items():
    print(f"=== {source} ===")
    for metric_name in metric_names:
        vals = source_scores[metric_name]
        print(f"  {metric_name}: {sum(vals) / len(vals):.3f}")

=== Overall ===
  faithfulness: 1.000
  groundedness: 0.841
  answer_relevancy: 0.891
  context_precision: 0.854
  context_recall: 0.791

=== manual ===
  faithfulness: 1.000
  groundedness: 0.918
  answer_relevancy: 0.972
  context_precision: 0.990
  context_recall: 0.880
=== synthetic ===
  faithfulness: 1.000
  groundedness: 0.799
  answer_relevancy: 0.846
  context_precision: 0.778
  context_recall: 0.742


## Export LightRAG results

In [11]:
lightrag_output = build_output(lightrag_results, lr_scores, lr_scores_by_source, metric_names)
with open("eval_results_lightrag.json", "w") as f:
    json.dump(lightrag_output, f, indent=2)

print(f"Wrote {len(lightrag_output['samples'])} sample results to eval_results_lightrag.json")

Wrote 14 sample results to eval_results_lightrag.json
